<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Módulo 4: Creación de variables binarias</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.15em; line-height: 1.3;">Ingeniería de Características</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Maestría en Ciencia de Datos</h3>
    </div>
</div>

Una variable categórica guarda una etiqueta, no una cantidad. Para usarla en muchos modelos, la representamos con números. La codificación debe respetar la información que sí existe: dos categorías pueden ser distintas sin que una sea mayor que la otra.

En esta guía veremos cómo representar una variable binaria con 0/1 y cómo extender esa idea a variables nominales con varias categorías. Compararemos las columnas generadas y su interpretación en regresión lineal.

## Variables indicadoras y ejemplo

Una variable **binaria** tiene exactamente dos valores posibles y puede representarse con una sola columna 0/1. En este notebook también usaremos `Ciudad`, que tiene varias categorías y no tiene un orden natural. La codificaremos con columnas indicadoras para comparar one-hot, codificación ficticia y codificación de efectos.

El conjunto de datos de ejemplo relaciona una ciudad con una renta generada para practicar la interpretación de los coeficientes. No representa una muestra real ni permite sacar conclusiones sobre precios de vivienda.

In [44]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [45]:
#Dataset de ejemplo
np.random.seed(1)
categorías=['Guadalajara','Zapopan','Ciudad de México','Monterrey','Morelia','Mexicali']
ponderación=[1/4,1/5,1/4,7/40,1/20,3/40]
limites=[(3500,6000),(3000,12000),(4000,18000),(3000,6000),(2000,5000),(3500,5000)]
ciudad=np.random.choice(categorías,size=20,p=ponderación)
rentas=[]
for c in ciudad:
    rentas.append(np.random.randint(*limites[categorías.index(c)]))
df=pd.DataFrame()
df['Rentas']=rentas
df['Ciudad']=ciudad
df.head()

,Rentas,Ciudad
0,9285,Zapopan
1,5025,Monterrey
2,4531,Guadalajara
3,7415,Zapopan
4,4837,Guadalajara


# Codificación One-Hot

## One-hot encoding

Crea una columna indicadora por categoría. Cada observación recibe `1` en la columna que le corresponde y `0` en las demás. Por ejemplo, para `Ciudad ∈ {Guadalajara, Zapopan, Morelia}`, una fila de Zapopan queda como `(0, 1, 0)`.

Es útil para variables nominales porque no asigna un orden artificial. El número de columnas crece con la cardinalidad: si hay $K$ categorías, se crean $K$ columnas. Para modelos lineales con intercepto, la suma de las columnas siempre es 1, por lo que incluirlas todas introduce redundancia; la codificación ficticia es una alternativa cuando se desea una categoría de referencia.

In [46]:
pd.get_dummies(df)                             # Codifica todas las variables categóricas

,Rentas,Ciudad_Ciudad de México,Ciudad_Guadalajara,Ciudad_Monterrey,Ciudad_Morelia,Ciudad_Zapopan
0,9285,False,False,False,False,True
1,5025,False,False,True,False,False
2,4531,False,True,False,False,False
3,7415,False,False,False,False,True
4,4837,False,True,False,False,False
5,5420,False,True,False,False,False
6,3816,False,True,False,False,False
7,11920,False,False,False,False,True
8,9771,False,False,False,False,True
9,4431,True,False,False,False,False


In [47]:
pd.get_dummies(df['Ciudad'])                  # No agrega el prefijo

,Ciudad de México,Guadalajara,Monterrey,Morelia,Zapopan
0,False,False,False,False,True
1,False,False,True,False,False
2,False,True,False,False,False
3,False,False,False,False,True
4,False,True,False,False,False
5,False,True,False,False,False
6,False,True,False,False,False
7,False,False,False,False,True
8,False,False,False,False,True
9,True,False,False,False,False


In [48]:
pd.get_dummies(df,columns=['Ciudad'])         # Codifica solo 'Ciudad' (Agrega el prefijo)

,Rentas,Ciudad_Ciudad de México,Ciudad_Guadalajara,Ciudad_Monterrey,Ciudad_Morelia,Ciudad_Zapopan
0,9285,False,False,False,False,True
1,5025,False,False,True,False,False
2,4531,False,True,False,False,False
3,7415,False,False,False,False,True
4,4837,False,True,False,False,False
5,5420,False,True,False,False,False
6,3816,False,True,False,False,False
7,11920,False,False,False,False,True
8,9771,False,False,False,False,True
9,4431,True,False,False,False,False


In [49]:
help(pd.get_dummies)

Help on function get_dummies in module pandas.core.reshape.encoding:

get_dummies(data, prefix=None, prefix_sep: 'str | Iterable[str] | dict[str, str]' = '_', dummy_na: 'bool' = False, columns=None, sparse: 'bool' = False, drop_first: 'bool' = False, dtype: 'NpDtype | None' = None) -> 'DataFrame'
    Convert categorical variable into dummy/indicator variables.

    Each variable is converted in as many 0/1 variables as there are different
    values. Columns in the output are each named after a value; if the input is
    a DataFrame, the name of the original variable is prepended to the value.

    Parameters
    ----------
    data : array-like, Series, or DataFrame
        Data of which to get dummy indicators.
    prefix : str, list of str, or dict of str, default None
        String to append DataFrame column names.
        Pass a list with length equal to the number of columns
        when calling get_dummies on a DataFrame. Alternatively, `prefix`
        can be a dictionary mapp

In [50]:
OH=pd.get_dummies(df,prefix='',prefix_sep='') # Especificamos el prefijo
OH.head(2)

,Rentas,Ciudad de México,Guadalajara,Monterrey,Morelia,Zapopan
0,9285,False,False,False,False,True
1,5025,False,False,True,False,False


### One-hot encoding con Scikit-learn

`OneHotEncoder` de scikit-learn es útil cuando la codificación forma parte del preprocesamiento de un modelo. `fit` aprende las categorías disponibles en los datos de entrenamiento; `transform` usa ese mismo esquema para convertir los datos. Separar ambas operaciones ayuda a evitar inconsistencias entre entrenamiento y evaluación.

En el ejemplo siguiente usamos `sparse_output=False` para mostrar una matriz densa pequeña. Para conjuntos con muchas categorías conviene conservar la salida dispersa. En un flujo de producción, configura también `handle_unknown` para decidir qué hacer cuando aparezca una categoría no observada durante el ajuste.

In [51]:
#from sklearn.preprocessing import OneHotEncoder 
from sklearn.preprocessing import OneHotEncoder

df_ejemplo = pd.DataFrame({'Color': ['Rojo', 'Azul', 'Verde', 'Azul', 'Rojo']})
encoder = OneHotEncoder(sparse_output=False)
onehot = encoder.fit_transform(df_ejemplo[['Color']])
df_onehot_sklearn = pd.DataFrame(onehot, columns=encoder.get_feature_names_out(['Color']))
print(df_onehot_sklearn)

   Color_Azul  Color_Rojo  Color_Verde
0         0.0         1.0          0.0
1         1.0         0.0          0.0
2         0.0         0.0          1.0
3         1.0         0.0          0.0
4         0.0         1.0          0.0


In [ ]:
# Ejemplo: conjunto de aprobación de crédito
data = pd.read_csv("Data/credit_approval_uci.csv")
data.head()

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,target
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,f,g,202.0,0,1
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,f,g,43.0,560,1
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,f,g,280.0,824,1
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,t,g,100.0,3,1
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,f,s,120.0,0,1


In [53]:
encoder = OneHotEncoder(sparse_output=False)

In [54]:
vars_categorical = data.select_dtypes(include="O").columns.to_list()
vars_categorical

['A1', 'A4', 'A5', 'A6', 'A7', 'A9', 'A10', 'A12', 'A13']

In [55]:
encoder.fit(data[vars_categorical])

,categories,'auto'
,drop,None
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [56]:
encoder.categories_

[array(['Missing', 'a', 'b'], dtype=object),
 array(['Missing', 'l', 'u', 'y'], dtype=object),
 array(['Missing', 'g', 'gg', 'p'], dtype=object),
 array(['Missing', 'aa', 'c', 'cc', 'd', 'e', 'ff', 'i', 'j', 'k', 'm',
        'q', 'r', 'w', 'x'], dtype=object),
 array(['Missing', 'bb', 'dd', 'ff', 'h', 'j', 'n', 'o', 'v', 'z'],
       dtype=object),
 array(['f', 't'], dtype=object),
 array(['f', 't'], dtype=object),
 array(['f', 't'], dtype=object),
 array(['g', 'p', 's'], dtype=object)]

In [57]:
data_enc =  encoder.transform(data[vars_categorical])
data_enc

array([[0., 0., 1., ..., 1., 0., 0.],
       [0., 1., 0., ..., 1., 0., 0.],
       [0., 1., 0., ..., 1., 0., 0.],
       ...,
       [0., 1., 0., ..., 1., 0., 0.],
       [0., 0., 1., ..., 1., 0., 0.],
       [0., 0., 1., ..., 1., 0., 0.]])

In [58]:
encoder.get_feature_names_out()

array(['A1_Missing', 'A1_a', 'A1_b', 'A4_Missing', 'A4_l', 'A4_u', 'A4_y',
       'A5_Missing', 'A5_g', 'A5_gg', 'A5_p', 'A6_Missing', 'A6_aa',
       'A6_c', 'A6_cc', 'A6_d', 'A6_e', 'A6_ff', 'A6_i', 'A6_j', 'A6_k',
       'A6_m', 'A6_q', 'A6_r', 'A6_w', 'A6_x', 'A7_Missing', 'A7_bb',
       'A7_dd', 'A7_ff', 'A7_h', 'A7_j', 'A7_n', 'A7_o', 'A7_v', 'A7_z',
       'A9_f', 'A9_t', 'A10_f', 'A10_t', 'A12_f', 'A12_t', 'A13_g',
       'A13_p', 'A13_s'], dtype=object)

In [59]:
#Transformamos a un dataframe
data_enc = pd.DataFrame(data_enc)
# agregamos el nombre de las variables categoricas encodeadas
data_enc.columns = encoder.get_feature_names_out()
data_enc.head()

,A1_Missing,A1_a,A1_b,A4_Missing,A4_l,A4_u,A4_y,A5_Missing,A5_g,A5_gg,...,A7_z,A9_f,A9_t,A10_f,A10_t,A12_f,A12_t,A13_g,A13_p,A13_s
0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
4,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0


In [60]:
data_transform = pd.concat([data, data_enc], axis=1)
# Drop variables categoricas
data_transform.drop(labels=vars_categorical, axis=1, inplace=True)
data_transform


,A2,A3,A8,A11,A14,A15,target,A1_Missing,A1_a,A1_b,...,A7_z,A9_f,A9_t,A10_f,A10_t,A12_f,A12_t,A13_g,A13_p,A13_s
0,30.83,0.000,1.25,1,202.0,0,1,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
1,58.67,4.460,3.04,6,43.0,560,1,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
2,24.50,0.500,1.50,0,280.0,824,1,0.0,1.0,0.0,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
3,27.83,1.540,3.75,5,100.0,3,1,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
4,20.17,5.625,1.71,0,120.0,0,1,0.0,0.0,1.0,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685,21.08,10.085,1.25,0,260.0,0,0,0.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
686,22.67,0.750,2.00,2,200.0,394,0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
687,25.25,13.500,2.00,1,200.0,1,0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
688,17.92,0.205,0.04,0,280.0,750,0,0.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0


### Ventajas y límites de one-hot

One-hot conserva la identidad de las categorías sin asignarles un orden. Es compatible con muchos modelos, incluidos modelos lineales y redes neuronales.

Cada categoría añade una columna. Cuando hay muchas categorías, la matriz puede ser grande y dispersa; considera la cardinalidad antes de elegir esta técnica. En modelos lineales con intercepto, incluir todas las columnas puede generar redundancia perfecta. La codificación ficticia elimina una columna de referencia para evitarla.

Los valores faltantes requieren una decisión explícita: imputarlos, tratarlos como una categoría propia o usar las opciones del codificador. Una fila con ceros en todas las columnas no identifica automáticamente un dato faltante; también podría representar una categoría desconocida o una categoría de referencia, según el método.

# Codificación Ficticia

La codificación ficticia conserva una columna menos que one-hot: una categoría se elige como **referencia** y queda representada por ceros en las columnas restantes. Para $K$ categorías genera $K-1$ columnas. `drop_first=True` omite una categoría según el orden interno de pandas; revisa ese orden o define la referencia de manera explícita si importa para la interpretación.

En una regresión lineal, el intercepto corresponde a la predicción para la categoría de referencia y cada coeficiente representa la diferencia respecto a ella. Para una variable binaria, basta una columna: el coeficiente compara la categoría codificada como 1 con la codificada como 0.

In [61]:
F=pd.get_dummies(df,prefix='',prefix_sep='',drop_first=True)
F.head(2)

,Rentas,Guadalajara,Monterrey,Morelia,Zapopan
0,9285,False,False,False,True
1,5025,False,True,False,False


In [62]:
df

,Rentas,Ciudad
0,9285,Zapopan
1,5025,Monterrey
2,4531,Guadalajara
3,7415,Zapopan
4,4837,Guadalajara
5,5420,Guadalajara
6,3816,Guadalajara
7,11920,Zapopan
8,9771,Zapopan
9,4431,Ciudad de México


In [63]:
np.where(df['Ciudad']=='Ciudad de México')

(array([ 9, 11, 15, 17], dtype=int64),)

In [64]:
F.iloc[[ 9, 11, 15, 17], :]

,Rentas,Guadalajara,Monterrey,Morelia,Zapopan
9,4431,False,False,False,False
11,16921,False,False,False,False
15,5478,False,False,False,False
17,16803,False,False,False,False


In [65]:
CR,=set(OH.columns)-set(F.columns)
CR

'Ciudad de México'

### Codificación ficticia con scikit-learn

`OneHotEncoder(drop='first')` aplica la misma idea de referencia. El codificador aprende las categorías al ajustarse con los datos de entrenamiento y reutiliza ese esquema al transformar nuevos datos. Esto resulta especialmente útil en pipelines y permite configurar el tratamiento de categorías no vistas.

In [73]:
encoder = OneHotEncoder(sparse_output=False, drop='first')
vars_categorical = data.select_dtypes(include="O").columns.to_list()
encoder.fit(data[vars_categorical])

,categories,'auto'
,drop,'first'
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [74]:
encoder.categories_

[array(['Missing', 'a', 'b'], dtype=object),
 array(['Missing', 'l', 'u', 'y'], dtype=object),
 array(['Missing', 'g', 'gg', 'p'], dtype=object),
 array(['Missing', 'aa', 'c', 'cc', 'd', 'e', 'ff', 'i', 'j', 'k', 'm',
        'q', 'r', 'w', 'x'], dtype=object),
 array(['Missing', 'bb', 'dd', 'ff', 'h', 'j', 'n', 'o', 'v', 'z'],
       dtype=object),
 array(['f', 't'], dtype=object),
 array(['f', 't'], dtype=object),
 array(['f', 't'], dtype=object),
 array(['g', 'p', 's'], dtype=object)]

In [75]:
data_enc =  encoder.transform(data[vars_categorical])

In [76]:
#Transformamos a un dataframe
data_enc = pd.DataFrame(data_enc)
data_enc.columns = encoder.get_feature_names_out()
data_transform = pd.concat([data, data_enc], axis=1)
data_transform.drop(labels=vars_categorical, axis=1, inplace=True)
data_transform

,A2,A3,A8,A11,A14,A15,target,A1_a,A1_b,A4_l,...,A7_j,A7_n,A7_o,A7_v,A7_z,A9_t,A10_t,A12_t,A13_p,A13_s
0,30.83,0.000,1.25,1,202.0,0,1,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0
1,58.67,4.460,3.04,6,43.0,560,1,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
2,24.50,0.500,1.50,0,280.0,824,1,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,27.83,1.540,3.75,5,100.0,3,1,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0
4,20.17,5.625,1.71,0,120.0,0,1,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685,21.08,10.085,1.25,0,260.0,0,0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
686,22.67,0.750,2.00,2,200.0,394,0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
687,25.25,13.500,2.00,1,200.0,1,0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
688,17.92,0.205,0.04,0,280.0,750,0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


# Codificación de Efectos

En codificación de efectos también se omite una columna, pero la categoría de referencia se representa con `-1` en las columnas restantes. Así, cada coeficiente de un modelo lineal se interpreta como una desviación respecto a la media general; la predicción de la categoría de referencia se obtiene combinando el intercepto y los coeficientes.

Por ejemplo, con tres categorías y `C` como referencia, `A = (1, 0)`, `B = (0, 1)` y `C = (-1, -1)`. El vector de referencia hace que los coeficientes se interpreten en relación con la media general, en vez de comparar cada nivel con `C`.

Este ejemplo construye esa representación manualmente para hacer visible el cambio. En proyectos reales conviene usar una implementación probada y comprobar cuál es la categoría de referencia.

In [ ]:
E = pd.get_dummies(df, prefix='', prefix_sep='', drop_first=True).astype(int)
CDMX, = np.where(df['Ciudad'] == CR)
CDMX

array([ 9, 11, 15, 17], dtype=int64)

In [78]:
E.iloc[CDMX,1:]=-1
E.head(10)

C:\Users\uie70742\AppData\Local\Temp\ipykernel_13528\3673512813.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-1' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  E.iloc[CDMX,1:]=-1
C:\Users\uie70742\AppData\Local\Temp\ipykernel_13528\3673512813.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-1' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  E.iloc[CDMX,1:]=-1
C:\Users\uie70742\AppData\Local\Temp\ipykernel_13528\3673512813.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-1' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  E.iloc[CDMX,1:]=-1
C:\Users\uie70742\AppData\Local\Temp\ipykernel_13528\3673512813.py:1: 

,Rentas,Guadalajara,Monterrey,Morelia,Zapopan
0,9285,False,False,False,True
1,5025,False,True,False,False
2,4531,True,False,False,False
3,7415,False,False,False,True
4,4837,True,False,False,False
5,5420,True,False,False,False
6,3816,True,False,False,False
7,11920,False,False,False,True
8,9771,False,False,False,True
9,4431,-1,-1,-1,-1


In [79]:
E.dtypes

Rentas          int64
Guadalajara    object
Monterrey      object
Morelia        object
Zapopan        object
dtype: object

In [80]:
E=E.astype('int')
E.dtypes

Rentas         int32
Guadalajara    int32
Monterrey      int32
Morelia        int32
Zapopan        int32
dtype: object

In [81]:
E.iloc[CDMX,1:]=-1
E.head(10)

,Rentas,Guadalajara,Monterrey,Morelia,Zapopan
0,9285,0,0,0,1
1,5025,0,1,0,0
2,4531,1,0,0,0
3,7415,0,0,0,1
4,4837,1,0,0,0
5,5420,1,0,0,0
6,3816,1,0,0,0
7,11920,0,0,0,1
8,9771,0,0,0,1
9,4431,-1,-1,-1,-1


# Regresión lineal

En una regresión lineal, una variable binaria codificada como 0/1 permite interpretar su coeficiente como la diferencia entre la predicción del grupo 1 y la del grupo 0, manteniendo las demás variables constantes.

Para una variable con varias categorías, el significado del intercepto y de los coeficientes depende del esquema: con referencia, cada coeficiente compara un nivel con esa referencia; con efectos, las diferencias se expresan respecto a la media general. La codificación cambia la parametrización y la interpretación, no la información original de las categorías.

### Reg lineal con OneHot

In [82]:
OH.head()

,Rentas,Ciudad de México,Guadalajara,Monterrey,Morelia,Zapopan
0,9285,False,False,False,False,True
1,5025,False,False,True,False,False
2,4531,False,True,False,False,False
3,7415,False,False,False,False,True
4,4837,False,True,False,False,False


In [83]:
linOH=LinearRegression()
linOH.fit(OH.drop(columns='Rentas'),OH.Rentas)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [84]:
linOH.coef_

array([ 4525.88333333, -1799.11666667, -1357.36666667, -3287.36666667,
        1917.96666667])

In [85]:
linOH.intercept_

6382.366666666668

$$ \hat{y} = a1*x_1 + a2*x_2 + ... + a5*x_5 + a0$$

In [86]:
promedios=df.groupby('Ciudad').mean()
promedios

,Rentas
Ciudad,
Ciudad de México,10908.250000
Guadalajara,4583.250000
Monterrey,5025.000000
Morelia,3095.000000
Zapopan,8300.333333


In [87]:
linOH.intercept_-promedios.mean()

Rentas    9.094947e-13
dtype: float64

In [88]:
promedios-promedios.mean()

,Rentas
Ciudad,
Ciudad de México,4525.883333
Guadalajara,-1799.116667
Monterrey,-1357.366667
Morelia,-3287.366667
Zapopan,1917.966667


In [89]:
linOH.coef_

array([ 4525.88333333, -1799.11666667, -1357.36666667, -3287.36666667,
        1917.96666667])

**Para la codificación One-Hot, en una regresión lineal, la intersección es el promedio de los valores promedios de las rentas de cada categoría. Los coeficientes son el resultante de la resta entre el promedio de las categorías y el promedio general.**

### Reg lineal con Codificación Ficticia

In [90]:
linF=LinearRegression()
linF.fit(F.drop(columns='Rentas'),F.Rentas)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [91]:
CR

'Ciudad de México'

In [92]:
linF.intercept_-promedios.loc[CR]

Rentas   -1.818989e-12
Name: Ciudad de México, dtype: float64

In [93]:
linF.intercept_

10908.249999999998

In [94]:
promedios.loc[CR]

Rentas    10908.25
Name: Ciudad de México, dtype: float64

In [95]:
promedios-promedios.loc[CR]

,Rentas
Ciudad,
Ciudad de México,0.000000
Guadalajara,-6325.000000
Monterrey,-5883.250000
Morelia,-7813.250000
Zapopan,-2607.916667


In [96]:
linF.coef_

array([-6325.        , -5883.25      , -7813.25      , -2607.91666667])

**En la codificación Ficticia, la intersección de la regresión lineal corresponde al promedio de la categoría de referencia. Los coeficientes son la diferencia entre el promedio de cada categoría y la intersección.**

### Reg lineal con Codificación de Efectos

In [97]:
linE=LinearRegression()
linE.fit(E.drop(columns='Rentas'),E.Rentas)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [98]:
linOH.intercept_,linE.intercept_

(6382.366666666668, 6382.366666666667)

In [99]:
linOH.coef_[1:],linE.coef_

(array([-1799.11666667, -1357.36666667, -3287.36666667,  1917.96666667]),
 array([-1799.11666667, -1357.36666667, -3287.36666667,  1917.96666667]))

In [100]:
linE.intercept_+linE.coef_

array([4583.25      , 5025.        , 3095.        , 8300.33333333])

In [101]:
promedios

,Rentas
Ciudad,
Ciudad de México,10908.250000
Guadalajara,4583.250000
Monterrey,5025.000000
Morelia,3095.000000
Zapopan,8300.333333


In [102]:
# Promedio de la Categoría de referencia
linE.intercept_-linE.coef_.sum()

10908.25

**Para la codificación de Efectos, en una regresión lineal, la intersección es el promedio general y los coeficientes, la diferencia entre los promedios de cada variable y el promedio general (a esto se le conoce como efecto principal).**

### ¿Cuándo usar codificación de efectos?

En modelos lineales y análisis de varianza, esta codificación permite comparar los niveles con la media general. No es universalmente mejor que one-hot o la codificación ficticia: cambia la interpretación del intercepto y de los coeficientes, pero no añade información nueva. Elige el esquema según la pregunta que quieras responder y verifica cómo trata tu librería la categoría de referencia.

## Comparación de métodos de codificación

| Método | Columnas para $K$ categorías | Cómo se representa la referencia | Interpretación típica | Útil cuando... |
|---|---:|---|---|---|
| Binaria 0/1 | 1 | 0 representa una categoría y 1 la otra | Diferencia entre las dos categorías | La variable tiene exactamente dos niveles |
| One-hot | $K$ | No hay referencia omitida | Cada columna identifica un nivel | Se quiere conservar la identidad de categorías nominales |
| Ficticia | $K-1$ | Vector de ceros | Cada coeficiente compara con el nivel de referencia | Se interpretan coeficientes de un modelo lineal |
| Efectos | $K-1$ | -1 en todas las columnas | Coeficientes como desviaciones respecto a la media general | Interesa comparar niveles con el promedio global |

**Idea clave:** el mismo predictor puede producir matrices distintas según el esquema. Elige la codificación antes de interpretar los coeficientes y verifica siempre cuál es la categoría de referencia.

## Ejercicio aplicado: aprobación de crédito

Usa `Data/credit_approval_uci.csv`. El dataset contiene predictores `A1` a `A15` y una variable de respuesta `target`. Mantén `target` separado: no debe formar parte de las columnas que vas a codificar.

1. Revisa cardinalidad y faltantes. ¿Qué columnas son binarias y cuáles tienen más de dos categorías?
2. Codifica `A9` como binaria. Explica qué categoría decidiste representar con 1 y por qué.
3. Codifica `A6` con one-hot y con codificación ficticia. Compara el número de columnas y muestra algunas filas.
4. Explica cómo representaste los valores faltantes y cuál sería la categoría de referencia en el caso ficticio.
5. ¿Qué problemas tendrías si al predecir apareciera una categoría que no estaba en el entrenamiento?
